# Separe períodos e prepare as matrizes

Retome o [ajuste em treino](https://devrgs.github.io/Aprendendo-linguagens/#/python/preparacao-ml/ajustar-e-aplicar). Use a versão auditada do marco anterior, fornecida nos arquivos da atividade. Ordene por data e ID. Datas até 6 de janeiro de 2026 são treino, até 8 de janeiro são validação, e as demais são teste. A coluna minutos_faturados é posterior à chegada. Identifique o vazamento antes de construir X.

A mediana dos km observados apenas no treino é 4. Distâncias ausentes recebem 4. Os extremos após preenchimento são 2 e 8. Subtraia 2 e divida por 6, sem recortar. V07 contém 100 km e pode ter resultado acima de 1. Se a amplitude do treino for zero, a coluna escalada toda vale zero. Se não houver km observados no treino, interrompa com mensagem. Mantenha paradas na escala original. Conserve os IDs e as datas para rastrear as linhas.

Execute as células na ordem com **Shift+Enter**. A preparação cria uma pasta nova na área de arquivos do caderno. Baixe o caderno editado e os resultados que quiser guardar.


In [ ]:
from pathlib import Path as _Path
from tempfile import mkdtemp as _mkdtemp
import os as _os
import sys as _sys
from importlib import invalidate_caches as _refresh_imports

# Auxiliares editáveis devem ser relidos sem aproveitar bytecode antigo.
_sys.dont_write_bytecode = True

_raiz = _Path.cwd()
_trabalho = _Path(_mkdtemp(prefix='python-preparar-', dir=_raiz))
import numpy
import pandas
import matplotlib


## E1 · Preveja e execute

Construa partições cronológicas e confira as chaves em cada conjunto.

Se editar um auxiliar no painel de arquivos, salve a edição. Execute novamente a preparação desta etapa e a célula do exercício. A preparação inicial do caderno cria outra pasta.


In [ ]:
_pasta = _trabalho / 'e1'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_arquivo = _pasta / 'auditoria_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import json\n\nimport numpy as np\nimport pandas as pd\n\nTIPOS = {\n    "id": "string",\n    "data": "string",\n    "estacao": "string",\n    "km_planejados": "float64",\n    "paradas_planejadas": "Int64",\n    "minutos_reais": "float64",\n    "minutos_faturados": "float64",\n}\n\nUNIDADES = {\n    "id": "chave de viagem",\n    "data": "dia (AAAA-MM-DD)",\n    "estacao": "chave de estação",\n    "km_planejados": "km",\n    "paradas_planejadas": "paradas (contagem)",\n    "minutos_reais": "min",\n    "minutos_faturados": "min",\n}\n\n\ndef ler_viagens(caminho):\n    viagens = pd.read_csv(caminho, dtype=TIPOS, keep_default_na=False, na_values=[""])\n    if set(viagens.columns) != set(TIPOS):\n        raise ValueError("colunas de viagens diferentes do esquema")\n    if viagens[["id", "data", "estacao"]].isna().any().any():\n        raise ValueError("id, data e estacao devem ser preenchidos")\n    if not viagens["data"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}").all():\n        raise ValueError("data deve estar no formato AAAA-MM-DD")\n    viagens["data"] = pd.to_datetime(viagens["data"], format="%Y-%m-%d")\n    return viagens\n\n\ndef ler_estacoes(caminho):\n    estacoes = pd.read_csv(caminho, dtype="string", keep_default_na=False, na_values=[""])\n    if set(estacoes.columns) != {"estacao", "nome"} or estacoes.isna().any().any():\n        raise ValueError("estacoes.csv deve ter chave e nome preenchidos")\n    return estacoes\n\n\ndef auditar(brutas, estacoes):\n    duplicatas = brutas.loc[brutas.duplicated(), "id"].value_counts().sort_index()\n    unicas = brutas.drop_duplicates()\n    conflitos = sorted(unicas.loc[unicas["id"].duplicated(keep=False), "id"].unique())\n    if conflitos:\n        prefixo = "ID em conflito: " if len(conflitos) == 1 else "IDs em conflito: "\n        raise ValueError(prefixo + ", ".join(conflitos))\n\n    regras = {\n        "km_planejados negativo": unicas["km_planejados"].lt(0),\n        "km_planejados infinito": np.isinf(unicas["km_planejados"]),\n        "paradas_planejadas ausente": unicas["paradas_planejadas"].isna(),\n        "paradas_planejadas negativo": unicas["paradas_planejadas"].lt(0).fillna(False),\n        "minutos_reais ausente": unicas["minutos_reais"].isna(),\n        "minutos_reais negativo": unicas["minutos_reais"].lt(0),\n        "minutos_reais infinito": np.isinf(unicas["minutos_reais"]),\n    }\n    rejeitadas = {}\n    for regra, mascara in regras.items():\n        for chave in sorted(unicas.loc[mascara, "id"]):\n            rejeitadas.setdefault(chave, []).append(regra)\n    validas = unicas.loc[~unicas["id"].isin(rejeitadas)].copy()\n    validas["paradas_planejadas"] = validas["paradas_planejadas"].astype("int64")\n    uniao = validas.merge(estacoes, on="estacao", how="left", validate="many_to_one", indicator=True)\n    auditoria = {\n        "linhas_brutas": len(brutas),\n        "linhas_validas": len(uniao),\n        "duplicatas_excedentes": {chave: int(n) for chave, n in duplicatas.items()},\n        "total_duplicatas_excedentes": int(duplicatas.sum()),\n        "rejeitadas": rejeitadas,\n        "total_rejeitadas": len(rejeitadas),\n        "km_ausentes_validas": sorted(validas.loc[validas["km_planejados"].isna(), "id"]),\n        "viagens_sem_cadastro": sorted(uniao.loc[uniao["_merge"] == "left_only", "id"]),\n        "decisoes": [\n            "Conservar o original e remover só cópias de linhas idênticas.",\n            "Interromper quando registros diferentes usam o mesmo id.",\n            "Conservar km ausente para imputação no preparo.",\n            "Conservar zero paradas e distâncias altas dentro do domínio.",\n            "Manter em auditoria as viagens sem cadastro de estação.",\n        ],\n        "unidades": UNIDADES,\n    }\n    return uniao.drop(columns="_merge"), auditoria\n\n\ndef salvar_json(caminho, dados):\n    caminho.write_text(\n        json.dumps(dados, ensure_ascii=False, indent=2, sort_keys=True) + "\\n",\n        encoding="utf-8",\n    )\n', encoding='utf-8')
_arquivo = _pasta / 'preparacao_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('from statistics import median\n\nimport numpy as np\nimport pandas as pd\n\nATRIBUTOS = ["km_escalados", "paradas_planejadas"]\nALVO = "minutos_reais"\nFIM_TREINO = pd.Timestamp("2026-01-06")\nFIM_VALIDACAO = pd.Timestamp("2026-01-08")\n\n\ndef particionar(validas):\n    ordenadas = validas.sort_values(["data", "id"])\n    datas = ordenadas["data"]\n    partes = {\n        "treino": ordenadas.loc[datas <= FIM_TREINO],\n        "validacao": ordenadas.loc[(datas > FIM_TREINO) & (datas <= FIM_VALIDACAO)],\n        "teste": ordenadas.loc[datas > FIM_VALIDACAO],\n    }\n    if any(parte.empty for parte in partes.values()):\n        raise ValueError("treino, validação e teste devem ter pelo menos uma viagem")\n    return partes\n\n\ndef ajustar(treino):\n    km = treino["km_planejados"]\n    observados = km.dropna()\n    if observados.empty:\n        raise ValueError("treino sem distâncias observadas")\n    mediana = float(observados.median())\n    preenchidos = km.fillna(mediana)\n    return {\n        "km": {\n            "mediana": mediana,\n            "minimo": float(preenchidos.min()),\n            "maximo": float(preenchidos.max()),\n        },\n        "referencia_minutos": float(median(treino[ALVO])),\n        "atributos": ATRIBUTOS.copy(),\n        "alvo": ALVO,\n        "convencao_amplitude_zero": "km_escalados vale 0 em todas as partições",\n        "fronteiras_inclusivas": {\n            "fim_treino": FIM_TREINO.date().isoformat(),\n            "fim_validacao": FIM_VALIDACAO.date().isoformat(),\n        },\n        "instante_previsao": "antes da partida",\n        "fora_de_X": {\n            "id": "chave de rastreio",\n            "data": "define partições e fica para rastreio",\n            "estacao": "chave para auditoria",\n            "nome": "nome para auditoria",\n            "minutos_reais": "alvo conhecido depois da chegada",\n            "minutos_faturados": "calculado após a chegada",\n        },\n    }\n\n\ndef aplicar(parte, parametros):\n    escala = parametros["km"]\n    km = parte["km_planejados"].fillna(escala["mediana"])\n    amplitude = escala["maximo"] - escala["minimo"]\n    escalados = (km - escala["minimo"]) / amplitude if amplitude else pd.Series(0.0, index=parte.index)\n    return parte.assign(km_escalados=escalados)[["id", *ATRIBUTOS, ALVO]].reset_index(drop=True)\n\n\ndef extrair(tabela):\n    X = tabela[ATRIBUTOS].to_numpy(dtype=float)\n    y = tabela[ALVO].to_numpy(dtype=float)\n    ids = tabela["id"].to_numpy(dtype=str)\n    return X, y, ids\n\n\ndef exportar(partes, parametros, saida):\n    tabelas = {}\n    rastreio = []\n    for nome, parte in partes.items():\n        tabela = aplicar(parte, parametros)\n        tabelas[nome] = tabela\n        tabela.to_csv(saida / f"{nome}.csv", index=False)\n        rastreio.append(parte[["id", "data"]].assign(particao=nome))\n    pd.concat(rastreio).to_csv(saida / "rastreio.csv", index=False, date_format="%Y-%m-%d")\n    return tabelas\n\n\ndef mae(alvos, previsao):\n    if len(alvos) == 0:\n        raise ValueError("MAE requer ao menos um alvo")\n    return float(np.abs(np.asarray(alvos, dtype=float) - previsao).mean())\n', encoding='utf-8')
_refresh_imports()


In [ ]:
from pathlib import Path
from auditoria_viagens import auditar, ler_estacoes, ler_viagens

brutas = ler_viagens(Path("dados/viagens.csv"))
estacoes = ler_estacoes(Path("dados/estacoes.csv"))
validas, auditoria = auditar(brutas, estacoes)

from preparacao_viagens import ajustar, aplicar, exportar, extrair, particionar
partes = particionar(validas)
parametros = ajustar(partes["treino"])

for nome, parte in partes.items():
    print(nome, parte["id"].tolist())
ids = [set(p["id"]) for p in partes.values()]
assert all(not a & b for i, a in enumerate(ids) for b in ids[i+1:])


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
treino ['V01', 'V02', 'V03', 'V04', 'V05', 'V06']
validacao ['V07', 'V08']
teste ['V09', 'V10']
```

Os limites de cada etapa são inclusivos no lado final, com os conjuntos disjuntos. Os IDs servem para conferir a separação, e a data determina a ordem.


## E2 · Altere ou depure

Em treino apenas, copie faturamento como se fosse uma previsão. Meça o erro e explique por que não usar a coluna.

Se editar um auxiliar no painel de arquivos, salve a edição. Execute novamente a preparação desta etapa e a célula do exercício. A preparação inicial do caderno cria outra pasta.


In [ ]:
_pasta = _trabalho / 'e2'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_arquivo = _pasta / 'auditoria_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import json\n\nimport numpy as np\nimport pandas as pd\n\nTIPOS = {\n    "id": "string",\n    "data": "string",\n    "estacao": "string",\n    "km_planejados": "float64",\n    "paradas_planejadas": "Int64",\n    "minutos_reais": "float64",\n    "minutos_faturados": "float64",\n}\n\nUNIDADES = {\n    "id": "chave de viagem",\n    "data": "dia (AAAA-MM-DD)",\n    "estacao": "chave de estação",\n    "km_planejados": "km",\n    "paradas_planejadas": "paradas (contagem)",\n    "minutos_reais": "min",\n    "minutos_faturados": "min",\n}\n\n\ndef ler_viagens(caminho):\n    viagens = pd.read_csv(caminho, dtype=TIPOS, keep_default_na=False, na_values=[""])\n    if set(viagens.columns) != set(TIPOS):\n        raise ValueError("colunas de viagens diferentes do esquema")\n    if viagens[["id", "data", "estacao"]].isna().any().any():\n        raise ValueError("id, data e estacao devem ser preenchidos")\n    if not viagens["data"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}").all():\n        raise ValueError("data deve estar no formato AAAA-MM-DD")\n    viagens["data"] = pd.to_datetime(viagens["data"], format="%Y-%m-%d")\n    return viagens\n\n\ndef ler_estacoes(caminho):\n    estacoes = pd.read_csv(caminho, dtype="string", keep_default_na=False, na_values=[""])\n    if set(estacoes.columns) != {"estacao", "nome"} or estacoes.isna().any().any():\n        raise ValueError("estacoes.csv deve ter chave e nome preenchidos")\n    return estacoes\n\n\ndef auditar(brutas, estacoes):\n    duplicatas = brutas.loc[brutas.duplicated(), "id"].value_counts().sort_index()\n    unicas = brutas.drop_duplicates()\n    conflitos = sorted(unicas.loc[unicas["id"].duplicated(keep=False), "id"].unique())\n    if conflitos:\n        prefixo = "ID em conflito: " if len(conflitos) == 1 else "IDs em conflito: "\n        raise ValueError(prefixo + ", ".join(conflitos))\n\n    regras = {\n        "km_planejados negativo": unicas["km_planejados"].lt(0),\n        "km_planejados infinito": np.isinf(unicas["km_planejados"]),\n        "paradas_planejadas ausente": unicas["paradas_planejadas"].isna(),\n        "paradas_planejadas negativo": unicas["paradas_planejadas"].lt(0).fillna(False),\n        "minutos_reais ausente": unicas["minutos_reais"].isna(),\n        "minutos_reais negativo": unicas["minutos_reais"].lt(0),\n        "minutos_reais infinito": np.isinf(unicas["minutos_reais"]),\n    }\n    rejeitadas = {}\n    for regra, mascara in regras.items():\n        for chave in sorted(unicas.loc[mascara, "id"]):\n            rejeitadas.setdefault(chave, []).append(regra)\n    validas = unicas.loc[~unicas["id"].isin(rejeitadas)].copy()\n    validas["paradas_planejadas"] = validas["paradas_planejadas"].astype("int64")\n    uniao = validas.merge(estacoes, on="estacao", how="left", validate="many_to_one", indicator=True)\n    auditoria = {\n        "linhas_brutas": len(brutas),\n        "linhas_validas": len(uniao),\n        "duplicatas_excedentes": {chave: int(n) for chave, n in duplicatas.items()},\n        "total_duplicatas_excedentes": int(duplicatas.sum()),\n        "rejeitadas": rejeitadas,\n        "total_rejeitadas": len(rejeitadas),\n        "km_ausentes_validas": sorted(validas.loc[validas["km_planejados"].isna(), "id"]),\n        "viagens_sem_cadastro": sorted(uniao.loc[uniao["_merge"] == "left_only", "id"]),\n        "decisoes": [\n            "Conservar o original e remover só cópias de linhas idênticas.",\n            "Interromper quando registros diferentes usam o mesmo id.",\n            "Conservar km ausente para imputação no preparo.",\n            "Conservar zero paradas e distâncias altas dentro do domínio.",\n            "Manter em auditoria as viagens sem cadastro de estação.",\n        ],\n        "unidades": UNIDADES,\n    }\n    return uniao.drop(columns="_merge"), auditoria\n\n\ndef salvar_json(caminho, dados):\n    caminho.write_text(\n        json.dumps(dados, ensure_ascii=False, indent=2, sort_keys=True) + "\\n",\n        encoding="utf-8",\n    )\n', encoding='utf-8')
_arquivo = _pasta / 'preparacao_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('from statistics import median\n\nimport numpy as np\nimport pandas as pd\n\nATRIBUTOS = ["km_escalados", "paradas_planejadas"]\nALVO = "minutos_reais"\nFIM_TREINO = pd.Timestamp("2026-01-06")\nFIM_VALIDACAO = pd.Timestamp("2026-01-08")\n\n\ndef particionar(validas):\n    ordenadas = validas.sort_values(["data", "id"])\n    datas = ordenadas["data"]\n    partes = {\n        "treino": ordenadas.loc[datas <= FIM_TREINO],\n        "validacao": ordenadas.loc[(datas > FIM_TREINO) & (datas <= FIM_VALIDACAO)],\n        "teste": ordenadas.loc[datas > FIM_VALIDACAO],\n    }\n    if any(parte.empty for parte in partes.values()):\n        raise ValueError("treino, validação e teste devem ter pelo menos uma viagem")\n    return partes\n\n\ndef ajustar(treino):\n    km = treino["km_planejados"]\n    observados = km.dropna()\n    if observados.empty:\n        raise ValueError("treino sem distâncias observadas")\n    mediana = float(observados.median())\n    preenchidos = km.fillna(mediana)\n    return {\n        "km": {\n            "mediana": mediana,\n            "minimo": float(preenchidos.min()),\n            "maximo": float(preenchidos.max()),\n        },\n        "referencia_minutos": float(median(treino[ALVO])),\n        "atributos": ATRIBUTOS.copy(),\n        "alvo": ALVO,\n        "convencao_amplitude_zero": "km_escalados vale 0 em todas as partições",\n        "fronteiras_inclusivas": {\n            "fim_treino": FIM_TREINO.date().isoformat(),\n            "fim_validacao": FIM_VALIDACAO.date().isoformat(),\n        },\n        "instante_previsao": "antes da partida",\n        "fora_de_X": {\n            "id": "chave de rastreio",\n            "data": "define partições e fica para rastreio",\n            "estacao": "chave para auditoria",\n            "nome": "nome para auditoria",\n            "minutos_reais": "alvo conhecido depois da chegada",\n            "minutos_faturados": "calculado após a chegada",\n        },\n    }\n\n\ndef aplicar(parte, parametros):\n    escala = parametros["km"]\n    km = parte["km_planejados"].fillna(escala["mediana"])\n    amplitude = escala["maximo"] - escala["minimo"]\n    escalados = (km - escala["minimo"]) / amplitude if amplitude else pd.Series(0.0, index=parte.index)\n    return parte.assign(km_escalados=escalados)[["id", *ATRIBUTOS, ALVO]].reset_index(drop=True)\n\n\ndef extrair(tabela):\n    X = tabela[ATRIBUTOS].to_numpy(dtype=float)\n    y = tabela[ALVO].to_numpy(dtype=float)\n    ids = tabela["id"].to_numpy(dtype=str)\n    return X, y, ids\n\n\ndef exportar(partes, parametros, saida):\n    tabelas = {}\n    rastreio = []\n    for nome, parte in partes.items():\n        tabela = aplicar(parte, parametros)\n        tabelas[nome] = tabela\n        tabela.to_csv(saida / f"{nome}.csv", index=False)\n        rastreio.append(parte[["id", "data"]].assign(particao=nome))\n    pd.concat(rastreio).to_csv(saida / "rastreio.csv", index=False, date_format="%Y-%m-%d")\n    return tabelas\n\n\ndef mae(alvos, previsao):\n    if len(alvos) == 0:\n        raise ValueError("MAE requer ao menos um alvo")\n    return float(np.abs(np.asarray(alvos, dtype=float) - previsao).mean())\n', encoding='utf-8')
_refresh_imports()


In [ ]:
from pathlib import Path
from auditoria_viagens import auditar, ler_estacoes, ler_viagens

brutas = ler_viagens(Path("dados/viagens.csv"))
estacoes = ler_estacoes(Path("dados/estacoes.csv"))
validas, auditoria = auditar(brutas, estacoes)

from preparacao_viagens import ajustar, aplicar, exportar, extrair, particionar
partes = particionar(validas)
parametros = ajustar(partes["treino"])

treino = partes["treino"]
erro = (treino["minutos_faturados"] - treino["minutos_reais"]).abs().mean()
print(f"{erro:.1f} min")
print(parametros["atributos"])


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
0.0 min
['km_escalados', 'paradas_planejadas']
```

Faturamento coincide com duração real nestes dados. O resultado usa informação posterior, indisponível antes da partida.


## E3 · Resolva

Exporte as três partições e parametros.json. Confira formas, ordem das colunas e os controles de V01, V03, V07, V08, V09 e V10.

Se editar um auxiliar no painel de arquivos, salve a edição. Execute novamente a preparação desta etapa e a célula do exercício. A preparação inicial do caderno cria outra pasta.


In [ ]:
_pasta = _trabalho / 'e3'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_arquivo = _pasta / 'auditoria_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import json\n\nimport numpy as np\nimport pandas as pd\n\nTIPOS = {\n    "id": "string",\n    "data": "string",\n    "estacao": "string",\n    "km_planejados": "float64",\n    "paradas_planejadas": "Int64",\n    "minutos_reais": "float64",\n    "minutos_faturados": "float64",\n}\n\nUNIDADES = {\n    "id": "chave de viagem",\n    "data": "dia (AAAA-MM-DD)",\n    "estacao": "chave de estação",\n    "km_planejados": "km",\n    "paradas_planejadas": "paradas (contagem)",\n    "minutos_reais": "min",\n    "minutos_faturados": "min",\n}\n\n\ndef ler_viagens(caminho):\n    viagens = pd.read_csv(caminho, dtype=TIPOS, keep_default_na=False, na_values=[""])\n    if set(viagens.columns) != set(TIPOS):\n        raise ValueError("colunas de viagens diferentes do esquema")\n    if viagens[["id", "data", "estacao"]].isna().any().any():\n        raise ValueError("id, data e estacao devem ser preenchidos")\n    if not viagens["data"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}").all():\n        raise ValueError("data deve estar no formato AAAA-MM-DD")\n    viagens["data"] = pd.to_datetime(viagens["data"], format="%Y-%m-%d")\n    return viagens\n\n\ndef ler_estacoes(caminho):\n    estacoes = pd.read_csv(caminho, dtype="string", keep_default_na=False, na_values=[""])\n    if set(estacoes.columns) != {"estacao", "nome"} or estacoes.isna().any().any():\n        raise ValueError("estacoes.csv deve ter chave e nome preenchidos")\n    return estacoes\n\n\ndef auditar(brutas, estacoes):\n    duplicatas = brutas.loc[brutas.duplicated(), "id"].value_counts().sort_index()\n    unicas = brutas.drop_duplicates()\n    conflitos = sorted(unicas.loc[unicas["id"].duplicated(keep=False), "id"].unique())\n    if conflitos:\n        prefixo = "ID em conflito: " if len(conflitos) == 1 else "IDs em conflito: "\n        raise ValueError(prefixo + ", ".join(conflitos))\n\n    regras = {\n        "km_planejados negativo": unicas["km_planejados"].lt(0),\n        "km_planejados infinito": np.isinf(unicas["km_planejados"]),\n        "paradas_planejadas ausente": unicas["paradas_planejadas"].isna(),\n        "paradas_planejadas negativo": unicas["paradas_planejadas"].lt(0).fillna(False),\n        "minutos_reais ausente": unicas["minutos_reais"].isna(),\n        "minutos_reais negativo": unicas["minutos_reais"].lt(0),\n        "minutos_reais infinito": np.isinf(unicas["minutos_reais"]),\n    }\n    rejeitadas = {}\n    for regra, mascara in regras.items():\n        for chave in sorted(unicas.loc[mascara, "id"]):\n            rejeitadas.setdefault(chave, []).append(regra)\n    validas = unicas.loc[~unicas["id"].isin(rejeitadas)].copy()\n    validas["paradas_planejadas"] = validas["paradas_planejadas"].astype("int64")\n    uniao = validas.merge(estacoes, on="estacao", how="left", validate="many_to_one", indicator=True)\n    auditoria = {\n        "linhas_brutas": len(brutas),\n        "linhas_validas": len(uniao),\n        "duplicatas_excedentes": {chave: int(n) for chave, n in duplicatas.items()},\n        "total_duplicatas_excedentes": int(duplicatas.sum()),\n        "rejeitadas": rejeitadas,\n        "total_rejeitadas": len(rejeitadas),\n        "km_ausentes_validas": sorted(validas.loc[validas["km_planejados"].isna(), "id"]),\n        "viagens_sem_cadastro": sorted(uniao.loc[uniao["_merge"] == "left_only", "id"]),\n        "decisoes": [\n            "Conservar o original e remover só cópias de linhas idênticas.",\n            "Interromper quando registros diferentes usam o mesmo id.",\n            "Conservar km ausente para imputação no preparo.",\n            "Conservar zero paradas e distâncias altas dentro do domínio.",\n            "Manter em auditoria as viagens sem cadastro de estação.",\n        ],\n        "unidades": UNIDADES,\n    }\n    return uniao.drop(columns="_merge"), auditoria\n\n\ndef salvar_json(caminho, dados):\n    caminho.write_text(\n        json.dumps(dados, ensure_ascii=False, indent=2, sort_keys=True) + "\\n",\n        encoding="utf-8",\n    )\n', encoding='utf-8')
_arquivo = _pasta / 'preparacao_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('from statistics import median\n\nimport numpy as np\nimport pandas as pd\n\nATRIBUTOS = ["km_escalados", "paradas_planejadas"]\nALVO = "minutos_reais"\nFIM_TREINO = pd.Timestamp("2026-01-06")\nFIM_VALIDACAO = pd.Timestamp("2026-01-08")\n\n\ndef particionar(validas):\n    ordenadas = validas.sort_values(["data", "id"])\n    datas = ordenadas["data"]\n    partes = {\n        "treino": ordenadas.loc[datas <= FIM_TREINO],\n        "validacao": ordenadas.loc[(datas > FIM_TREINO) & (datas <= FIM_VALIDACAO)],\n        "teste": ordenadas.loc[datas > FIM_VALIDACAO],\n    }\n    if any(parte.empty for parte in partes.values()):\n        raise ValueError("treino, validação e teste devem ter pelo menos uma viagem")\n    return partes\n\n\ndef ajustar(treino):\n    km = treino["km_planejados"]\n    observados = km.dropna()\n    if observados.empty:\n        raise ValueError("treino sem distâncias observadas")\n    mediana = float(observados.median())\n    preenchidos = km.fillna(mediana)\n    return {\n        "km": {\n            "mediana": mediana,\n            "minimo": float(preenchidos.min()),\n            "maximo": float(preenchidos.max()),\n        },\n        "referencia_minutos": float(median(treino[ALVO])),\n        "atributos": ATRIBUTOS.copy(),\n        "alvo": ALVO,\n        "convencao_amplitude_zero": "km_escalados vale 0 em todas as partições",\n        "fronteiras_inclusivas": {\n            "fim_treino": FIM_TREINO.date().isoformat(),\n            "fim_validacao": FIM_VALIDACAO.date().isoformat(),\n        },\n        "instante_previsao": "antes da partida",\n        "fora_de_X": {\n            "id": "chave de rastreio",\n            "data": "define partições e fica para rastreio",\n            "estacao": "chave para auditoria",\n            "nome": "nome para auditoria",\n            "minutos_reais": "alvo conhecido depois da chegada",\n            "minutos_faturados": "calculado após a chegada",\n        },\n    }\n\n\ndef aplicar(parte, parametros):\n    escala = parametros["km"]\n    km = parte["km_planejados"].fillna(escala["mediana"])\n    amplitude = escala["maximo"] - escala["minimo"]\n    escalados = (km - escala["minimo"]) / amplitude if amplitude else pd.Series(0.0, index=parte.index)\n    return parte.assign(km_escalados=escalados)[["id", *ATRIBUTOS, ALVO]].reset_index(drop=True)\n\n\ndef extrair(tabela):\n    X = tabela[ATRIBUTOS].to_numpy(dtype=float)\n    y = tabela[ALVO].to_numpy(dtype=float)\n    ids = tabela["id"].to_numpy(dtype=str)\n    return X, y, ids\n\n\ndef exportar(partes, parametros, saida):\n    tabelas = {}\n    rastreio = []\n    for nome, parte in partes.items():\n        tabela = aplicar(parte, parametros)\n        tabelas[nome] = tabela\n        tabela.to_csv(saida / f"{nome}.csv", index=False)\n        rastreio.append(parte[["id", "data"]].assign(particao=nome))\n    pd.concat(rastreio).to_csv(saida / "rastreio.csv", index=False, date_format="%Y-%m-%d")\n    return tabelas\n\n\ndef mae(alvos, previsao):\n    if len(alvos) == 0:\n        raise ValueError("MAE requer ao menos um alvo")\n    return float(np.abs(np.asarray(alvos, dtype=float) - previsao).mean())\n', encoding='utf-8')
_refresh_imports()


In [ ]:
from pathlib import Path
from auditoria_viagens import auditar, ler_estacoes, ler_viagens

brutas = ler_viagens(Path("dados/viagens.csv"))
estacoes = ler_estacoes(Path("dados/estacoes.csv"))
validas, auditoria = auditar(brutas, estacoes)

from preparacao_viagens import ajustar, aplicar, exportar, extrair, particionar
partes = particionar(validas)
parametros = ajustar(partes["treino"])

from math import isclose
from auditoria_viagens import salvar_json
pasta = Path("saida")
pasta.mkdir(exist_ok=True)
salvar_json(pasta/"parametros.json", parametros)
tabelas = exportar(partes, parametros, pasta)
for nome, tabela in tabelas.items():
    X, y, ids = extrair(tabela)
    print(nome, X.shape, y.shape)
valores = {"V01":(0,0), "V03":(1/3,1), "V07":(98/6,1), "V08":(1/3,0), "V09":(.5,2), "V10":(7/6,3)}
for t in tabelas.values():
    for chave, km, paradas in t[["id","km_escalados","paradas_planejadas"]].itertuples(index=False):
        if chave in valores:
            a, b = valores[chave]
            assert isclose(km, a, rel_tol=0, abs_tol=1e-6) and paradas == b
print("controles conferidos")


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
treino (6, 2) (6,)
validacao (2, 2) (2,)
teste (2, 2) (2,)
controles conferidos
```

Os shapes preservam a contagem de viagens. As duas colunas de X têm ordem fixa. V07 e V10 ficam acima de 1, pois excedem o máximo do treino. V08 usa o preenchimento aprendido no treino e preserva zero paradas. Verifique um treino constante para testar a regra da amplitude.
